In [13]:
#import re
import pandas as pd
from pandas import DataFrame
import os
#import datetime
#import shutil
#import glob
import urllib
import urllib.request
from pathlib import Path
import csv
import re
from bs4 import BeautifulSoup as bs

def append_type(data, game_opponent,Type,Outcome,Team,play, home_nickname, visitor_nickname):
    Type.append(play)
    Outcome.append('')
    if home_nickname in data:
        Team.append(home_nickname)
    else:
        Team.append(visitor_nickname)
    
                
def makeHeaderPBP(sport, year1, school):
    sport_schedule = open(r'/Users/shelbyyoung/Desktop/sports analytics/SAA files/' + school+'/'+ sport + '/' + str(year1) + '_links.csv', 'r')
    sport_games = [line.rstrip('\n') for line in sport_schedule]
    sport_schedule.close()
    game = 1
    for game_url in sport_games:
        print(game_url)
        #print(game)
        list_game = game_url.split("/")
        skip = False
        try:
            try:
                r= urllib.request.urlopen(game_url)
                game_opponent = list_game[7]
                game_id= list_game[9]

                header = open(r'/Users/shelbyyoung/Desktop/sports analytics/SAA files/' +school +'/'+ sport + '/' + 'headers' + '/' +
                      str(year1) + '/'+ 'game' +str(game)+'_'+ game_opponent+'_'+str(game_id)+'.csv', 'r')

                
                #assigns first row of the header as list'header_titles' and second row as 'header_info'
                header_data = list(csv.reader(header)) 
                header_titles = header_data[0][1:]
                header_info = header_data[1][1:]
                path = '/Users/shelbyyoung/Desktop/sports analytics/SAA files/' +school +'/'+ sport + '/' + 'Tables' + '/' + str(year1) + '/'+ str(game_id)+'_'+ game_opponent+'.csv'
            
                if os.path.exists(path):
                    table = open(r'/Users/shelbyyoung/Desktop/sports analytics/SAA files/' +school +'/'+ sport + '/' + 
                        'Tables' + '/' + str(year1) + '/'+ str(game_id)+'_'+ game_opponent+'.csv', 'r')

                    #assigns the first row of the table as list'table_titles'
                    table_data = list(csv.reader(table))
                    table_titles = table_data[0][1:]

                    #puts all titles into a combined list
                    all_titles = header_titles
                    all_titles.extend(table_titles)

                    df = pd.DataFrame(columns = all_titles)

                    i = 1
                    for row in range(len(table_data)-1):
                        combined = header_info[:]
                        combined.extend(table_data[i][1:])
                        df.loc[len(df.index)] = combined
                        i+=1

                    #url = 'https://centrecolonels.com/sports/' + str(sport) + '/stats/' + str(year1) +'/'+ game_opponent + '/boxscore/'+ str(game_id)
                    #print(url)
                    
                   

                    soup = bs(r, 'html.parser')
                    table = soup.find('table',{'class':'sidearm-table play-by-play'})
                    
                    visitor_nickname = table.find_all('th',{'class':'text-center hide-on-medium-down'})[0].text
                    visitor_nickname = visitor_nickname.replace(' - Play Description', '')

                    home_nickname = table.find_all('th',{'class':'text-center hide-on-medium-down'})[1].text
                    home_nickname = home_nickname.replace(' - Play Description', '')
                    print(home_nickname)
                    #except IndexError:
                        #print('except')

                    #visitor_nickname, home_nickname = get_table_nicknames(sport,year1,game_opponent, game_id)
                    visitor_nickname = visitor_nickname.lower()
                    home_nickname = home_nickname.lower()
                    #print(centre_nickname)

                    df.head()
                    #loops through the dataFrame.
                    #going to use to loop through play data to create a new column
                    #called type.
                    Type = []
                    #outcome from play, whether it be substitution or a shot
                    Outcome= []
                    #what team did what
                    Team = []
                    #whether or not it was a faceoff
                    faceoff = []
                    #who has the ball
                    possession = []
                    
                    data_num = 0
                    
                    for data in df['Play']:
                        data = data.lower()
                        #print(data)
                        
                           
                        if 'faceoff' in data:
                            faceoff.append('1')
                            if 'ground ball pickup' in data:
                                if visitor_nickname in data:
                                    first_instance = data.find(visitor_nickname)
                                    name = visitor_nickname
                                elif home_nickname in data:
                                    first_instance = data.find(home_nickname)
                                    name = home_nickname
                        else:
                            faceoff.append('0')

                        
                            
                        if 'shot by' in data:
                            Type.append('shot')
                            index = data.find('shot by')
                            if 'save' in data:
                                Outcome.append('saved')
                            elif 'blocked' in data:
                                Outcome.append('blocked')
                            elif 'wide' in data:
                                Outcome.append('wide')
                            else:
                                Outcome.append('')

                            if home_nickname in data:
                                Team.append(home_nickname)
                            else:
                                Team.append(visitor_nickname)
                        elif 'substitution' in data:
                            Type.append('substitution')
                            index = data.find('substitution: ')
                            #Outcome.append(data[index+len('substitution:'):len(data)])
                            Outcome.append("")
                            if home_nickname in data:
                                Team.append(home_nickname)
                            else:
                                Team.append(visitor_nickname)

                        elif 'goal' in data and "goalie" not in data:
                            Type.append('goal')
                            Outcome.append('')

                            if home_nickname in data:
                                Team.append(home_nickname)
                            else:
                                Team.append(visitor_nickname)
                        elif 'timeout' in data:
                            append_type(data, game_opponent,Type,Outcome, Team,'timeout', home_nickname, visitor_nickname)
                        elif 'draw control' in data:
                            append_type(data, game_opponent,Type,Outcome, Team,'draw control',home_nickname, visitor_nickname)
                        elif 'turnover' in data:
                            append_type(data, game_opponent,Type,Outcome, Team,'turnover',home_nickname, visitor_nickname)
                #need to add what kind of card and who got it
                        elif 'card' in data:
                            Type.append('Penalty')
                            if 'red' in data:
                                Outcome.append('red card')
                            elif 'yellow' in data:
                                Outcome.append('yellow card')
                            elif 'green' in data:
                                Outcome.append('green card')
                            else:
                                Outcome.append("")
                            if home_nickname in data:
                                Team.append(home_nickname)
                            else:
                                Team.append(visitor_nickname)
                        elif 'clear attempt' in data:
                            Type.append('clear attempt')
                            if 'good' in data:
                                Outcome.append('good')
                            else:
                                Outcome.append('failed')
                            if home_nickname in data:
                                Team.append(home_nickname)
                            else:
                                Team.append(visitor_nickname)
                        elif 'ground ball pickup' in data:
                            append_type(data, game_opponent,Type,Outcome, Team,'ground ball pickup',home_nickname, visitor_nickname)
                        elif 'free position attempt' in data:
                            append_type(data, game_opponent,Type,Outcome, Team,'free position attempt',home_nickname, visitor_nickname)
                       
                        else:
                            Type.append('')
                            Team.append('')
                            Outcome.append('')
  
                        
                        if 'at goalie' not in data:
                            if home_nickname in data:

                                if ('turnover' not in data) and ('penalty' not in data):
                                    possession.append(home_nickname)
                                else:
                                    possession.append(game_opponent)
                            elif ('end-of-period' not in data) and (data != ''):
                                if ('turnover' not in data) and ('penalty' not in data):
                                    possession.append(visitor_nickname)
                                else:
                                    possession.append(home_nickname)
                            else:
                                possession.append('')
                        else:
                            possession.append('')

                        
                        #print(data_num, Outcome[data_num])
                        data_num = data_num+1
                        
                    if sport == 'mens-lacrosse':
                        df['Faceoff'] = faceoff
                    df['Play Type'] = Type
                    df['Play Outcome'] = Outcome
                    df['Team']= Team
                    df['Possession'] = possession
                    
                    
                    
                    time_in_secs = []
                    time_left = []
                    data_num = 0
                    for time in df['Time']:
                        time = time.replace('[','')
                        time = time.replace(']','')
                        if (':' in time) and ('GB' not in time):
                            minutes = int(time.split(':')[0])
                            seconds = int(time.split(':')[1])
                            total_seconds = minutes * 60 + seconds
                            time_in_secs.append(total_seconds)
                            
                            #doesn't account for overtime
                            if sport == 'womens-lacrosse':
                                period_num = int(df['period number'][data_num])
                                #if period_num == 3:
                                    #print('ot')
                                if (time_in_secs[1] == 0) and (game_opponent != 'indiana-tech'):
                                    seconds_left = 3600 - (total_seconds+1800*(period_num-1))
                                else:
                                    seconds_left = total_seconds + 1800*(2-period_num)
                                if period_num > 2:
                                    seconds_left = total_seconds
                                if url == 'https://centrecolonels.com/sports/womens-lacrosse/stats/2011/birmingham-southern/boxscore/2450':
                                    if period_num >2:  
                                        seconds_left = 180-total_seconds
                                
                                
                            if sport == 'mens-lacrosse':
                                period_num = int(df['period number'][data_num])
                                if period_num == 5:
                                    print('ot')
                                seconds_left = total_seconds + 900*(4-period_num)
                                if period_num > 4:
                                    seconds_left = total_seconds
                                
                            time_left.append(seconds_left)
                        else:
                            time_in_secs.append('')
                            time_left.append('')
                            
                        data_num += 1
                            
                        
                    #print(time_in_secs[1])    
                    df['time in seconds'] = time_in_secs
                    df['time left'] = time_left

                    
                    output_file = 'game'+str(game)+'_'+game_opponent+'_'+str(game_id)+'.csv'
                    output_dir = Path('/Users/shelbyyoung/Desktop/sports analytics/SAA files/'+school+'/' + sport + '/TableHeader/' + str(year1) + '/')
                    output_dir.mkdir(parents=True, exist_ok=True)
                    df.to_csv(output_dir / output_file)


                else:
                    print("Play by play does not exist for that game, but a header does")

                game= game +1
                
            #except (IndexError,AttributeError):
            except (AttributeError):
                print('except')
                game=game+1
                continue

        except urllib.error.HTTPError:
            continue
 
 
  
 
def main():
    #get_centre_table_name('field-hockey',2010,'wooster', 497)
    #for year in range(2011,2021):
     #   makeHeaderPBP('womens-lacrosse', year)
        
    lacrosse_schools = ['berryvikings', 'bscsports', 'hendrixwarriors', 'rhodeslynx', 'sewaneetigers']
    
    for school in lacrosse_schools: #change based on sport
        print('school: ', school)
        
        for year in range(2011,2012):
            makeHeaderPBP('mens-lacrosse', year, school)
            
    '''sports = ['baseball', 'mens-basketball', 'football', 'mens-lacrosse', 'mens-soccer', 'mens-tennis', 'womens-basketball',
          'field-hockey', 'womens-lacrosse', 'womens-soccer', 'softball', 'womens-tennis', 'womens-volleyball']
  
    multi_year_sports = ['mens-basketball', 'womens-basketball', 'mens-tennis', 'womens-tennis']
 
    fall_sports = ['mens-basketball', 'football', 'mens-soccer', 'womens-basketball', 'field-hockey', 'womens-soccer', 'womens-volleyball']
    for sport in sports:
        if sport == 'mens-tennis' or sport == 'womens-tennis':
            year1 = 2018
        elif sport in fall_sports:
            year1 = 2010
        else:
            year1 = 2011
 
        while year1 != 2022:
            #year2 = year1 + 1
            #if sport in multi_year_sports:
                #years = str(year1) + '-' + str(year2)
            #else:
                #years = year1
            makeHeaders(sport,year1)
            year1 += 1
 
'''
main()

school:  berryvikings
https://berryvikings.com/sports/mens-lacrosse/stats/2011/millsaps/boxscore/1101
BERRY
https://berryvikings.com/sports/mens-lacrosse/stats/2011/guilford/boxscore/1102
GC
https://berryvikings.com/sports/mens-lacrosse/stats/2011/birmingham-southern/boxscore/1103
BSC
https://berryvikings.com/sports/mens-lacrosse/stats/2011/catholic/boxscore/1104
Play by play does not exist for that game, but a header does
https://berryvikings.com/sports/mens-lacrosse/stats/2011/southwestern-tex-/boxscore/1105
SULAX
https://berryvikings.com/sports/mens-lacrosse/stats/2011/concordia-wi-/boxscore/1106
CUWM11
https://berryvikings.com/sports/mens-lacrosse/stats/2011/carthage/boxscore/1107
BC
https://berryvikings.com/sports/mens-lacrosse/stats/2011/centre/boxscore/1108
CEN
https://berryvikings.com/sports/mens-lacrosse/stats/2011/aurora/boxscore/1109
BC
https://berryvikings.com/sports/mens-lacrosse/stats/2011/ferrum/boxscore/1110
FCM
https://berryvikings.com/sports/mens-lacrosse/stats/2011/o